# Training a Potential: how to setup data pipeline, model and trainer


# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp


## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data.


In [1]:
%load_ext autoreload
%autoreload 2
import logging, sys
logger = logging.getLogger("molpot")
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(stream=sys.stdout))

import molpot as mpot
import torch
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path
from molpot import alias

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [2]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu",
    total=1000,
    processes=[mpot.process.NeighborList(cutoff=5.0)],
)

Parsing molecule aspirin


Loaded 1000 frames


In [ ]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 1000

                     dataset: rMD17                     
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ label  ┃      type       ┃    unit    ┃   comment    ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ energy │ <class 'float'> │  kcal/mol  │ total energy │
│ forces │ <class 'float'> │ kcal/mol/A │  all forces  │
└────────┴─────────────────┴────────────┴──────────────┘

In [4]:
train_ds, eval_ds = torch.utils.data.random_split(rmd17_ds, [0.95, 0.05])
train_dl = mpot.DataLoader(train_ds, batch_size=1)
eval_dl = mpot.DataLoader(eval_ds, batch_size=1)

## Setting up the model


In [6]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 5.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(5.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1,
)
e_readout = mpot.potential.nnp.readout.Atomwise(
    in_keys=[("pinet", "p1")],
    out_keys=[("predicts", "energy")],
    n_neurons=[64, 1],
    reduce="sum",
)
f_readout = mpot.potential.nnp.readout.Derivative(
    fx_key=("predicts", "energy"), dx_key=alias.pair_diff, out_keys=("predicts", "forces"), 
)
potential = mpot.potential.PotentialSeq(pinet, e_readout)
# potential.derivative = f_readout

save_dir = Path("pinet2-rmd17")

optimizer = torch.optim.Adam(potential.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
loss_fn = mpot.engine.loss.MultiTargetLoss(
    torch.nn.MSELoss(), [("energy", "energy", 1.0)]# , ("forces", "forces", 10.0)]
)

potential_trainer = mpot.PotentialTrainer(
    model=potential,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device="cpu",
    no_grad_eval=False,
)
# potential_trainer.add_lr_scheduler(scheduler)
potential_trainer.add_checkpoint(save_dir)
potential_trainer.enable_progressbar("trainer")
potential_trainer.enable_progressbar("evaluator")

potential_trainer.add_metric(
    "e_mae",
    MeanAbsoluteError(mpot.engine.metric.get_tensor("energy", "energy")),
    BatchWise(),
    engine="trainer",
)
potential_trainer.add_metric(
    "e_mae",
    MeanAbsoluteError(mpot.engine.metric.get_tensor("energy", "energy")),
    BatchWise(),
    engine="evaluator",
)
# potential_trainer.add_metric(
#     "f_mae",
#     MeanAbsoluteError(get_metrics("forces", "forces")),
#     BatchWise(),
#     target="all",
# )
potential_trainer.attach_tensorboard(
    save_dir / "tb",
)

state = potential_trainer.run(train_data=train_dl, max_epochs=3, eval_data=eval_dl)

/opt/conda/lib/python3.12/site-packages/ignite/handlers/tqdm_logger.py:127: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


KeyError: 'both'